# Modelling a glacier with IGM — Session 3: experiment deepening

In Session 2 you went through the **basic workflow** (data, calibration of the ice flow and of the melt, past, future, ELA retrieval) with the Rhone Glacier, then with your own glacier. Session 3 **deepens** some experiments on your glacier, by bringing new data and by customising IGM:

1. **The Little Ice Age extent**: compare the modelled glacier in 1850 with the mapped 1850 extent, and refine your calibration of the past
2. **The response time**: write / read your own IGM module (oscillating ELA)
3. **The moraines**: track debris with particles (if relevant for your glacier)
4. **Figures for the report**

Parts 1–3 are **suggestions** to deepen your study: you do **not** have to do all three, choose **at least one** (the most relevant for your glacier).

Keep working in **your** project `~/work/my-glacier` (copied in Session 2): do **not** copy the template again. Just update the course material (new plotting helpers) and add this notebook to your project, in a terminal:

```bash
cd ~/work/glamod && git pull
cp ~/work/glamod/session3/tutorial-session3.ipynb ~/work/my-glacier/
```

then open `~/work/my-glacier/tutorial-session3.ipynb` (it reads the runs of `~/work/my-glacier/outputs/`).

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

with open("experiment/params_past.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)
os.makedirs("figures", exist_ok=True)                  # figures for the report go here

## 1. The Little Ice Age extent: a closer calibration of the past

**Why?** In Session 2 you calibrated the melt (`melt_enhancer`) mostly on today's volume, and checked the LIA maximum by eye. The LIA maximum (~1850) is the best-documented past state of Alpine glaciers: its **moraines** are still visible, and it was mapped on the first topographic maps. Comparing the modelled extent in 1850 with the mapped one is a strong test of the model, and helps to constrain the **past climate** (pre-1901 `delta_temp`) and the **melt**.

**Resources for the 1850 extent**:
- **Swiss Glacier Inventory 1850** (SGI 1850, [GLAMOS](https://www.glamos.ch/en/downloads#inventories); Maisch et al., 2000): outlines of all Swiss glaciers in 1850, from historical maps and moraines. Downloaded automatically below (`data/sgi1850/`).
- **Historical maps** on [map.geo.admin.ch](https://map.geo.admin.ch) (*Journey through time*): Dufour map (1845–1865) and Siegfried map (1870–1926), close to the LIA maximum.
- **Moraines** on the relief (*swissALTI3D* hillshade) and aerial images (*SWISSIMAGE*) of map.geo.admin.ch: the LIA moraines are the youngest, fresh, unvegetated ridges in front of the glacier.
- **Length changes** since ~1880 (GLAMOS factsheet of your glacier, e.g. [Rhone Glacier](https://www.glamos.ch/en/factsheet#/B43-03)): the retreat of the front.

▶ First compare your calibrated past run (Session 2) with the 1850 outline:

In [ ]:
run = "outputs/params_past"                         # <- your calibrated past run (Session 2)
compare_lia_extent(run, INPUT, year=1850)
compare_runs([run])

**Then refine the calibration of the past**: copy `experiment/params_past.yaml` (e.g. `params_past_lia.yaml`, with the run folder `outputs/params_past_lia` at the end of the file), and adjust one parameter at a time:
- the glacier is **too small / too large in 1850** ➡️ colder / warmer pre-1901 `delta_temp` (rows 1700 and 1850 of `clim_trend_array`)
- the glacier is **too small / too large today** ➡️ more / less melt: `melt_enhancer`
- the front is too thin and advances too slowly / too fast ➡️ check the ice flow again (`arrhenius`, `tau_ref`, Session 2, section 2)

```bash
igm_run +experiment=params_past_lia
```

Aim for a run that reaches the 1850 outline **and** today's volume; no perfect fit is possible, **discuss** the remaining mismatch in your report. If your calibration changes, redo the future scenarios (`params_future.yaml`).

In [ ]:
run = "outputs/params_past_lia"
compare_lia_extent(run, INPUT, year=1850)
compare_runs(["outputs/params_past", run])

> ✅ **Calibration of the past, refined** (double-click to edit): pre-1901 `delta_temp` = ...  `melt_enhancer` = ...  (`arrhenius` = ...  `tau_ref` = ...)

## 2. Response time (your own IGM module, oscillating ELA)

**Why?** In Session 2 (ELA retrieval) you saw that the glacier does not react immediately to an ELA change. How long does it take? This **response time** tells how much of the present-day warming your glacier has not "seen" yet, and how much it will keep retreating even if the climate stabilises.

Here the ELA oscillates like a sine wave (period 200 yr, amplitude 200 m) and we look at how the glacier volume follows.

**Look at the module first**: this SMB is not part of IGM, it is a **user module** written in Python: `user/code/processes/smb_oscillating.py` (its parameters: `user/conf/processes/smb_oscillating.yaml`). Open it: like any IGM module, it has 3 functions, `initialize` (called once), `update` (at each time step: computes the ELA at time t, then the SMB from the surface elevation) and `finalize`. This is how you can extend IGM with your own processes.

▶ **In the terminal** (400 model years, a few minutes). For another period, copy `experiment/params_response_time.yaml` to `experiment/params_response_100.yaml`, set `period: 100` and the run folder `outputs/params_response_100` in it:

```bash
igm_run +experiment=params_response_time
igm_run +experiment=params_response_100
```

In [ ]:
plot_response_time("outputs/params_response_time")
# plot_response_time("outputs/params_response_100")

**What you should see**:
- The volume **lags behind** the ELA: the volume is maximal some decades **after** the ELA is lowest (e.g. ~45 years for the Rhone Glacier, period 200 yr).
- With a **shorter** period (50–100 yr), the glacier has no time to adjust: the volume oscillation is **small** and strongly delayed. With a **longer** period (400 yr), the glacier nearly follows the ELA: **large** oscillation.
- The lag gives an estimate of the **response time** of your glacier. Compare with $\tau \approx H / (-b_{\text{terminus}})$ (Lecture 1a; $H$: typical thickness, $b_{\text{terminus}}$: SMB at the front) and with the 1D tutorial of Session 1.

> ✅ **Conclusion — response time** (double-click to edit): the response time of my glacier is about ... years. Consequence for the future: by 2100, the glacier is still / no longer responding to the warming of today because ...

## 3. Moraines with particles (if relevant for your glacier)

**Why?** Rocks fallen on the accumulation area are buried, carried by the ice flow, and re-emerge in the ablation area, where they form **medial moraines** and **debris cover**; at the margins they build the **lateral and frontal moraines** that we use to map former extents (such as the LIA). IGM can track such debris with **particles** moved by the 3D ice flow. Most relevant for glaciers with visible medial moraines or debris cover (check aerial images on map.geo.admin.ch).

▶ **In the terminal**: put your calibrations (ice flow, `melt_enhancer`) in `experiment/params_past_moraines.yaml`, then

```bash
igm_run +experiment=params_past_moraines
```

In [ ]:
run_p = "outputs/params_past_moraines"
for year in [1850, 1950, 2020]:
    plot_particles(run_p, year=year)                # all particles, coloured by depth
plot_particles(run_p, year=2020, rh_min=0.95)       # debris at the surface only: where moraines form

## 4. Figures for the report

The helper functions in `utils/helper.py` are meant for quick looks. For the report, make your own figures: copy the code of a helper into a cell below and adapt it (years shown, titles, units, colors, fonts), then save it into `figures/` **before** `plt.show()`:

```python
plt.savefig("figures/fig_volume.png", dpi=200, bbox_inches="tight")
```

In [ ]:
# your figures here